# Análise Preliminar e Exploratória de Dados
## Dataset: Diabetes (Pima Indians)

**Disciplina:** Análise Inteligente de Dados  
**Curso:** Engenharia Biomédica - ISEC  

---

## Objetivos da Aula

1. Realizar **análise preliminar** dos dados
2. Executar **análise exploratória** para compreender padrões
3. Identificar características, anomalias e relações nos dados
4. Aplicar técnicas de visualização adequadas

---

## Contexto do Dataset

Este dataset contém informações de mulheres da etnia Pima (índios americanos) com pelo menos 21 anos de idade. Os dados incluem medições clínicas e laboratoriais para estudar a diabetes.

**Variáveis:**
- `Pregnancies`: Número de gravidezes
- `Glucose`: Concentração de glicose plasmática (mg/dL)
- `BloodPressure`: Pressão arterial diastólica (mm Hg)
- `SkinThickness`: Espessura da prega cutânea tricipital (mm)
- `Insulin`: Insulina sérica (μU/mL)
- `BMI`: Índice de massa corporal (kg/m²)
- `DiabetesPedigreeFunction`: Função de genealogia de diabetes
- `Age`: Idade (anos)
- `Outcome`: Classe (0 = não diabético, 1 = diabético)


## 1. Importação de Bibliotecas

In [ ]:
# Bibliotecas essenciais
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

# Configurações de visualização
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuração para exibir todas as colunas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

print("✓ Bibliotecas importadas com sucesso!")

## 2. Carregamento dos Dados

In [ ]:
# URL do dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

# Nomes das colunas
column_names = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Carregar dados
df = pd.read_csv(url, names=column_names)

print("✓ Dados carregados com sucesso!")
print(f"Dimensões do dataset: {df.shape[0]} linhas × {df.shape[1]} colunas")

---
# PARTE I - ANÁLISE PRELIMINAR
---

## 3. Primeiras Observações

In [ ]:
# Visualizar primeiras linhas
print("=== PRIMEIRAS 10 LINHAS ===")
df.head(10)

In [ ]:
# Visualizar últimas linhas
print("=== ÚLTIMAS 5 LINHAS ===")
df.tail()

In [ ]:
# Amostra aleatória
print("=== AMOSTRA ALEATÓRIA (10 registos) ===")
df.sample(10, random_state=42)

### 🤔 Questão para Reflexão

Observando as primeiras linhas, nota algum padrão ou valor que pareça estranho?

## 4. Informações Gerais sobre o Dataset

In [ ]:
# Informação geral
print("=== INFORMAÇÕES GERAIS ===")
df.info()

In [ ]:
# Tipos de dados
print("=== TIPOS DE DADOS ===")
print(df.dtypes)
print("\n=== CONTAGEM POR TIPO ===")
print(df.dtypes.value_counts())

### ✏️ Exercício 1

**Responda:**
1. Quantas variáveis numéricas existem no dataset?
2. Existem variáveis categóricas?
3. Todas as variáveis têm o tipo de dados correto?

## 5. Análise de Valores em Falta (Missing Values)

In [ ]:
# Contagem de valores em falta
print("=== VALORES EM FALTA ===")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Valores em Falta': missing,
    'Percentagem (%)': missing_pct
})

print(missing_df)
print(f"\nTotal de valores em falta no dataset: {df.isnull().sum().sum()}")

In [ ]:
# IMPORTANTE: Neste dataset, zeros podem representar valores em falta!
# Variáveis que biologicamente não podem ser zero:
biologicamente_invalidos = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

print("=== VALORES ZERO (POTENCIALMENTE INVÁLIDOS) ===")
for col in biologicamente_invalidos:
    zero_count = (df[col] == 0).sum()
    zero_pct = (zero_count / len(df)) * 100
    print(f"{col:20s}: {zero_count:3d} zeros ({zero_pct:.1f}%)")

In [ ]:
# Visualização de zeros como missing values
fig, ax = plt.subplots(figsize=(10, 6))

zero_counts = []
for col in biologicamente_invalidos:
    zero_counts.append((df[col] == 0).sum())

ax.bar(biologicamente_invalidos, zero_counts, color='coral')
ax.set_xlabel('Variável', fontsize=12)
ax.set_ylabel('Número de Zeros', fontsize=12)
ax.set_title('Valores Zero (Biologicamente Inválidos)', fontsize=14, fontweight='bold')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

### 🤔 Questão para Reflexão

Porque é que valores zero em Glucose, BloodPressure, BMI, etc. são problemáticos? O que estes zeros podem representar?

## 6. Estatísticas Descritivas

In [ ]:
# Estatísticas descritivas completas
print("=== ESTATÍSTICAS DESCRITIVAS ===")
df.describe().round(2)

In [ ]:
# Estatísticas adicionais
print("=== ESTATÍSTICAS ADICIONAIS ===")
stats_df = pd.DataFrame({
    'Mediana': df.describe().loc['50%'],
    'Variância': df.var(),
    'Desvio Padrão': df.std(),
    'Coef. Variação (%)': (df.std() / df.mean() * 100)
}).round(2)

print(stats_df)

### ✏️ Exercício 2

**Analise as estatísticas descritivas e responda:**
1. Qual variável tem maior variabilidade (coeficiente de variação)?
2. A média e mediana são muito diferentes em alguma variável? O que isso sugere?
3. Quais são os valores mínimos e máximos mais suspeitos?

## 7. Análise de Duplicados

In [ ]:
# Verificar duplicados
print("=== ANÁLISE DE DUPLICADOS ===")
duplicados = df.duplicated().sum()
print(f"Número de linhas duplicadas: {duplicados}")
print(f"Percentagem de duplicados: {(duplicados/len(df)*100):.2f}%")

if duplicados > 0:
    print("\nExemplo de linhas duplicadas:")
    print(df[df.duplicated(keep=False)].sort_values(by=list(df.columns)))

---
# PARTE II - ANÁLISE EXPLORATÓRIA
---

## 8. Análise da Variável Alvo (Outcome)

In [ ]:
# Distribuição da variável alvo
print("=== DISTRIBUIÇÃO DO OUTCOME ===")
outcome_counts = df['Outcome'].value_counts()
outcome_pct = df['Outcome'].value_counts(normalize=True) * 100

outcome_df = pd.DataFrame({
    'Contagem': outcome_counts,
    'Percentagem (%)': outcome_pct.round(2)
})
outcome_df.index = ['Não Diabético (0)', 'Diabético (1)']
print(outcome_df)

In [ ]:
# Visualização da distribuição
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colors = ['#2ecc71', '#e74c3c']
outcome_counts.plot(kind='bar', ax=ax1, color=colors)
ax1.set_title('Distribuição de Diabetes', fontsize=14, fontweight='bold')
ax1.set_xlabel('Outcome', fontsize=12)
ax1.set_ylabel('Número de Pacientes', fontsize=12)
ax1.set_xticklabels(['Não Diabético', 'Diabético'], rotation=0)
ax1.grid(axis='y', alpha=0.3)

# Gráfico de pizza
ax2.pie(outcome_counts, labels=['Não Diabético', 'Diabético'], autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 12})
ax2.set_title('Proporção de Diabetes', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 🤔 Questão para Reflexão

O dataset está balanceado? O que significa ter um dataset desbalanceado no contexto clínico?

## 9. Distribuições Univariadas

In [ ]:
# Histogramas de todas as variáveis
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    axes[idx].hist(df[col], bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'Distribuição de {col}', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)
    axes[idx].set_ylabel('Frequência', fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribuições com KDE (Kernel Density Estimation)
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    sns.histplot(df[col], kde=True, ax=axes[idx], color='teal', bins=30)
    axes[idx].set_title(f'{col} - Histograma + KDE', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel(col, fontsize=10)

plt.tight_layout()
plt.show()

### ✏️ Exercício 3

**Observando os histogramas, identifique:**
1. Quais variáveis seguem aproximadamente uma distribuição normal?
2. Quais variáveis são assimétricas (skewed)?
3. Onde é que os zeros artificiais são visíveis?

## 10. Análise de Outliers com Boxplots

In [ ]:
# Boxplots de todas as variáveis
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.ravel()

for idx, col in enumerate(df.columns):
    axes[idx].boxplot(df[col], vert=True, patch_artist=True,
                     boxprops=dict(facecolor='lightblue', color='navy'),
                     medianprops=dict(color='red', linewidth=2),
                     whiskerprops=dict(color='navy'),
                     capprops=dict(color='navy'))
    axes[idx].set_title(f'Boxplot de {col}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel(col, fontsize=10)
    axes[idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Contagem de outliers por variável (método IQR)
print("=== DETECÇÃO DE OUTLIERS (Método IQR) ===")
print("\nNúmero de outliers por variável:")

outliers_summary = {}
for col in df.columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    outliers_summary[col] = len(outliers)
    
    print(f"{col:25s}: {len(outliers):3d} outliers ({len(outliers)/len(df)*100:.1f}%)")

### 🤔 Questão para Reflexão

Um valor atípico (outlier) em dados biomédicos é sempre um erro? Quando pode ser clinicamente relevante?

## 11. Matriz de Correlação

In [ ]:
# Calcular matriz de correlação
correlation_matrix = df.corr()

print("=== MATRIZ DE CORRELAÇÃO ===")
print(correlation_matrix.round(2))

In [ ]:
# Heatmap da matriz de correlação
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação - Dataset Diabetes', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlações com Outcome
print("=== CORRELAÇÕES COM OUTCOME (Diabetes) ===")
outcome_corr = correlation_matrix['Outcome'].drop('Outcome').sort_values(ascending=False)
print(outcome_corr.round(3))

# Visualização
plt.figure(figsize=(10, 6))
outcome_corr.plot(kind='barh', color=['green' if x > 0 else 'red' for x in outcome_corr])
plt.title('Correlação das Variáveis com Outcome (Diabetes)', fontsize=14, fontweight='bold')
plt.xlabel('Coeficiente de Correlação', fontsize=12)
plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### ✏️ Exercício 4

**Analise a matriz de correlação:**
1. Qual variável está mais correlacionada com Outcome?
2. Existem variáveis fortemente correlacionadas entre si?
3. Alguma correlação te surpreende do ponto de vista biomédico?

## 12. Análise Bivariada - Scatterplots

In [ ]:
# Pairplot das principais variáveis
principais_vars = ['Glucose', 'BMI', 'Age', 'BloodPressure', 'Insulin', 'Outcome']
sns.pairplot(df[principais_vars], hue='Outcome', palette={0: '#2ecc71', 1: '#e74c3c'},
             plot_kws={'alpha': 0.6}, diag_kind='kde', corner=False)
plt.suptitle('Pairplot - Relações entre Variáveis Principais', y=1.01, fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Scatterplot: Glucose vs BMI
fig, ax = plt.subplots(figsize=(10, 6))

colors = {0: '#2ecc71', 1: '#e74c3c'}
for outcome in [0, 1]:
    subset = df[df['Outcome'] == outcome]
    label = 'Não Diabético' if outcome == 0 else 'Diabético'
    ax.scatter(subset['Glucose'], subset['BMI'], c=colors[outcome], 
               label=label, alpha=0.6, s=50, edgecolor='black', linewidth=0.5)

ax.set_xlabel('Glucose (mg/dL)', fontsize=12)
ax.set_ylabel('BMI (kg/m²)', fontsize=12)
ax.set_title('Relação entre Glucose e BMI', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 13. Comparação de Distribuições por Grupo (Outcome)

In [ ]:
# Boxplots comparativos
variaveis_numericas = df.columns[:-1]  # Todas exceto Outcome

fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.ravel()

for idx, col in enumerate(variaveis_numericas):
    df.boxplot(column=col, by='Outcome', ax=axes[idx], patch_artist=True)
    axes[idx].set_title(f'{col} por Outcome', fontsize=11, fontweight='bold')
    axes[idx].set_xlabel('Outcome (0=Não Diabético, 1=Diabético)', fontsize=9)
    axes[idx].set_ylabel(col, fontsize=10)
    
plt.suptitle('Comparação de Variáveis entre Diabéticos e Não Diabéticos', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots para variáveis principais
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.ravel()

vars_principais = ['Glucose', 'BMI', 'Age', 'BloodPressure', 'Insulin', 'Pregnancies']

for idx, col in enumerate(vars_principais):
    sns.violinplot(data=df, x='Outcome', y=col, ax=axes[idx], palette={0: '#2ecc71', 1: '#e74c3c'})
    axes[idx].set_title(f'Distribuição de {col}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Outcome (0=Não, 1=Diabético)', fontsize=10)
    axes[idx].set_ylabel(col, fontsize=10)

plt.suptitle('Violin Plots - Comparação de Distribuições', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 14. Estatísticas por Grupo

In [ ]:
# Estatísticas descritivas por Outcome
print("=== ESTATÍSTICAS POR OUTCOME ===")
print("\n--- NÃO DIABÉTICOS (Outcome = 0) ---")
print(df[df['Outcome'] == 0].describe().round(2))

print("\n--- DIABÉTICOS (Outcome = 1) ---")
print(df[df['Outcome'] == 1].describe().round(2))

In [ ]:
# Comparação de médias
print("=== COMPARAÇÃO DE MÉDIAS ===")
comparacao = pd.DataFrame({
    'Não Diabético (média)': df[df['Outcome'] == 0].mean(),
    'Diabético (média)': df[df['Outcome'] == 1].mean(),
    'Diferença': df[df['Outcome'] == 1].mean() - df[df['Outcome'] == 0].mean(),
    'Diferença (%)': ((df[df['Outcome'] == 1].mean() - df[df['Outcome'] == 0].mean()) / 
                      df[df['Outcome'] == 0].mean() * 100)
}).round(2)

print(comparacao)

## 15. Análise de Faixas Etárias

In [ ]:
# Criar faixas etárias
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 30, 40, 50, 60, 100],
                         labels=['21-30', '31-40', '41-50', '51-60', '60+'])

# Distribuição de diabetes por faixa etária
age_diabetes = pd.crosstab(df['Age_Group'], df['Outcome'], normalize='index') * 100

print("=== PREVALÊNCIA DE DIABETES POR FAIXA ETÁRIA (%) ===")
print(age_diabetes.round(2))

In [ ]:
# Visualização
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Gráfico de barras empilhadas
age_counts = pd.crosstab(df['Age_Group'], df['Outcome'])
age_counts.plot(kind='bar', stacked=True, ax=ax1, color=['#2ecc71', '#e74c3c'])
ax1.set_title('Contagem de Casos por Faixa Etária', fontsize=13, fontweight='bold')
ax1.set_xlabel('Faixa Etária', fontsize=11)
ax1.set_ylabel('Número de Casos', fontsize=11)
ax1.legend(['Não Diabético', 'Diabético'], fontsize=10)
ax1.tick_params(axis='x', rotation=0)

# Percentagem de diabetes por faixa
age_diabetes[1].plot(kind='bar', ax=ax2, color='#e74c3c')
ax2.set_title('Prevalência de Diabetes por Faixa Etária', fontsize=13, fontweight='bold')
ax2.set_xlabel('Faixa Etária', fontsize=11)
ax2.set_ylabel('% de Diabéticos', fontsize=11)
ax2.tick_params(axis='x', rotation=0)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 16. Síntese e Conclusões
---

### 📊 Principais Descobertas

**Análise Preliminar:**
- Dataset com 768 observações e 9 variáveis
- Não existem valores em falta explícitos (NaN)
- **PROBLEMA CRÍTICO:** Valores zero em variáveis biologicamente inválidas (Glucose, BloodPressure, BMI, etc.)
- Estes zeros representam provavelmente dados em falta

**Análise Exploratória:**
- Dataset ligeiramente desbalanceado: ~65% não diabéticos vs ~35% diabéticos
- **Glucose** é a variável mais correlacionada com diabetes (correlação positiva)
- **BMI** e **Age** também mostram correlação positiva com diabetes
- Prevalência de diabetes aumenta com a idade
- Outliers presentes em várias variáveis, mas podem ser clinicamente relevantes

**Diferenças entre Grupos:**
- Diabéticos têm valores médios mais elevados em praticamente todas as variáveis
- Diferença mais marcada em Glucose (+30%), Insulin (+77%) e BMI (+17%)

**Próximos Passos Sugeridos:**
1. Tratamento dos valores zero (imputação ou remoção)
2. Análise mais detalhada de outliers
3. Feature engineering (criação de novas variáveis)
4. Preparação para modelação preditiva

### ✏️ Exercício Final

**Reflexão Crítica:**

Com base em toda a análise exploratória realizada:

1. **Qualidade dos Dados:**
   - Qual é o principal problema de qualidade neste dataset?
   - Como proporia resolver os valores zero inválidos?

2. **Insights Clínicos:**
   - Que variáveis parecem ser mais importantes para identificar diabetes?
   - Os resultados estão alinhados com o conhecimento médico sobre diabetes?

3. **Limitações:**
   - Que limitações vê neste dataset?
   - Que informação adicional seria útil ter?

4. **Próximos Passos:**
   - Que análises adicionais gostaria de fazer?
   - Como prepararia estes dados para um modelo preditivo?

---
## 📚 Referências e Recursos

**Dataset:**
- UCI Machine Learning Repository - Pima Indians Diabetes Database
- Fonte: National Institute of Diabetes and Digestive and Kidney Diseases

**Documentação:**
- Pandas: https://pandas.pydata.org/
- Matplotlib: https://matplotlib.org/
- Seaborn: https://seaborn.pydata.org/
- SciPy: https://scipy.org/

---
**Engenharia Biomédica - IPC**  
*Análise Inteligente de Dados*